# Notebook 01: Environment Setup & Unity Catalog
**Exam Coverage**: Sections 1 (Databricks Lakehouse Platform) & 5 (Data Governance)
**Duration**: 30-45 minutes
---
## Learning Objectives
By the end of this notebook, you will be able to:
- Understand Unity Catalog's three-level namespace (catalog.schema.table)
- Create and manage catalogs, schemas, and volumes
- Create managed tables in Unity Catalog
- Implement access control using GRANT and REVOKE
- Understand serverless compute and its benefits
---

## Section 1: Introduction and Setup
Unity Catalog provides centralized governance for all data and AI assets in Databricks.
### Three-Level Namespace
```
catalog.schema.table
```
This structure enables:
- Logical separation of environments (dev, staging, prod)
- Organizational boundaries
- Fine-grained access control
Let's set up our environment.

Import shared variables and configuration

In [0]:
%run ./variables

# Configuration Variables

Central configuration file for the Databricks Data Engineer Certification Lab.

**Usage**: Import this file in all notebooks to maintain consistent naming.

```python
%run ./variables
```

## Unity Catalog Configuration

## Volume Paths

## Checkpoint Locations

## Table Names

...

## Browser Types

## Locations (US Cities)

## Helper Functions

## Validation

## Display Configuration Summary

In [0]:
# Set the current catalog and schema for this session
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

# Verify current context
print(f"Current Catalog: {spark.catalog.currentCatalog()}")
print(f"Current Schema: {spark.catalog.currentDatabase()}")

Current Catalog: cert_prep_catalog
Current Schema: `01_bronze`


## Section 2: Explore Unity Catalog Structure
Unity Catalog organizes data in a hierarchy. Let's explore what's been set up.

In [0]:
# Display all catalogs available to you
spark.sql("SHOW CATALOGS").show()

+-----------------+
|          catalog|
+-----------------+
|cert_prep_catalog|
|          samples|
|           system|
|        workspace|
+-----------------+



In [0]:
# Display all schemas in the current catalog
spark.sql(f"SHOW SCHEMAS IN {CATALOG_NAME}").show()

+------------------+
|      databaseName|
+------------------+
|        00_landing|
|         01_bronze|
|         02_silver|
|           03_gold|
|           _system|
|           default|
|      dlt_pipeline|
|information_schema|
+------------------+



In [0]:
# Explore the landing volume structure in the landing zone for sales data
display(dbutils.fs.ls(SALES_LANDING_PATH))

path,name,size,modificationTime
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_SUCCESS,_SUCCESS,0,1778780175000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_187016012375518090,_committed_187016012375518090,736,1778780173000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_3634413423913501299,_committed_3634413423913501299,744,1778780175000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_4356316400236721844,_committed_4356316400236721844,744,1778780156000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_458255428166583646,_committed_458255428166583646,736,1778780171000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_5217697913066241031,_committed_5217697913066241031,744,1778780164000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_5596626546366511068,_committed_5596626546366511068,744,1778780159000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_8083424683851267525,_committed_8083424683851267525,744,1778780161000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_8457016463532102938,_committed_8457016463532102938,744,1778780166000
dbfs:/Volumes/cert_prep_catalog/00_landing/sales/_committed_8795120970909606546,_committed_8795120970909606546,744,1778780169000


In [0]:
# List tables in the bronze schema
# It will probably show no tables - rerun it once we create something!

spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{BRONZE_SCHEMA}").show()

+-----------+-----------------+-----------+
|   database|        tableName|isTemporary|
+-----------+-----------------+-----------+
|`01_bronze`|customers_managed|      false|
|`01_bronze`|    customers_raw|      false|
|`01_bronze`|       events_raw|      false|
|`01_bronze`|     products_raw|      false|
|`01_bronze`|        sales_raw|      false|
+-----------+-----------------+-----------+



### Understanding Volumes
Volumes provide governed access to files (non-tabular data).
**Three-level namespace for volumes:**
```
catalog.schema.volume
```
**Types:**
- **Managed**: Unity Catalog manages lifecycle and storage
- **External**: Points to external cloud storage

## Section 3: Create Your First Managed Table
**Managed tables**: Unity Catalog manages both metadata AND data files.
- Drop the table → deletes metadata + data
- Storage location is automatic
We'll load customer data from the landing zone and create a managed table.

In [0]:
# Define the path to sample customer data
customers_landing_path = f"{LANDING_BASE_PATH}/customers/"

# Read JSON data from the landing zone
customers_df = spark.read.json(customers_landing_path)

# Display the schema and sample data
customers_df.printSchema()
display(customers_df.limit(10))

root
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- registration_date: string (nullable = true)



customer_id,email,first_name,last_name,location,loyalty_tier,phone,registration_date
f0bd4810-2e5c-472b-8cb0-eb598c23d835,lisa.miller.3825@email.com,Lisa,Miller,"Detroit, MI, USA",Platinum,+1-530-370-7011,2025-10-09
df23a90d-142b-413f-b287-de7f3bea989d,patricia.martinez.3826@email.com,Patricia,Martinez,"Columbus, OH, USA",Silver,null,2026-05-03
ed31a79d-45f8-4322-8548-55fd68e1a989,emily.martinez.3827@email.com,Emily,Martinez,"Atlanta, GA, USA",Bronze,+1-417-676-2896,2026-03-22
1d3ce36e-180e-463a-acae-0b82ed0fec58,emily.rodriguez.3828@email.com,Emily,Rodriguez,"Detroit, MI, USA",Silver,+1-940-428-1215,2024-12-07
46f1f835-3749-4da9-a354-3e29d843562a,sarah.garcia.3829@email.com,null,Garcia,"Chicago, IL, USA",Gold,+1-336-571-9228,2024-06-11
5464b7a7-1238-47d1-83c3-89e1238dca38,lisa.lopez.3830@email.com,Lisa,Lopez,"San Jose, CA, USA",Bronze,+1-717-948-3017,2025-11-13
b7ae8894-f30b-4386-8ddb-0b298de83ebb,jennifer.garcia.3831@email.com,Jennifer,gARCIA,"San Antonio, TX, USA",Bronze,+1-271-403-5253,2025-05-27
c7435fb0-3fec-427c-94be-7bd95730d2df,david.garcia.3832@email.com,David,Garcia,"Miami, FL, USA",null,+1-862-614-5864,2024-08-20
f29239b9-749a-4038-b517-976cef2acce3,sarah.martinez.3833@email.com,sARAH,Martinez,"Denver, CO, USA",Bronze,+1-869-289-3482,2026-04-19
4dde769b-8bbf-4e03-8716-634b6c7c0032,jane.gonzalez.3834@email.com,Jane,Gonzalez,"Fort Worth, TX, USA",Bronze,null,2024-06-16


---
### 🎯 EXERCISE 1: Create a Managed Table
**Your task**: Save the `customers_df` DataFrame as a managed Delta table.
**Requirements:**
- Table name: `{CATALOG_NAME}.{BRONZE_SCHEMA}.customers_managed`
- Format: Delta
- Mode: Overwrite (since this is the first load)
- Use `.saveAsTable()` to create a managed table
**Key syntax:**
```python
df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("catalog.schema.table")
```
**Hint**: The table name variable `managed_table_name` is created for you below.

In [0]:
# TODO: Create a managed table

# Table name (provided)
managed_table_name = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.customers_managed"

# Write your code here:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(managed_table_name)

# SQL Version of the above 
# customers_df.createOrReplaceTempView("customers_df") # This is because its currently a dataframe and needs to be a temp view for sql to read it.
# spark.sql("""
# CREATE OR REPLACE TABLE managed_table_name
# AS
# SELECT * FROM customers_df;""")
# """



print(f"Created managed table: {managed_table_name}")

Created managed table: cert_prep_catalog.01_bronze.customers_managed


---
**Check the solution below** ⬇️

In [0]:
# ✅ SOLUTION: Create Managed Table

# managed_table_name = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.customers_managed"

# customers_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable(managed_table_name)

# print(f"✅ Created managed table: {managed_table_name}")

In [0]:
# Check table details to see where data is stored
display(spark.sql(f"DESCRIBE DETAIL {managed_table_name}"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,215f1a87-fb16-4b80-8d32-fdbe015e65d1,cert_prep_catalog.01_bronze.customers_managed,null,,2026-05-14T18:07:59.280Z,2026-05-20T19:57:53.000Z,List(),List(),1,369853,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


**Observation**: Notice the `location` field. For managed tables, Unity Catalog automatically determines this location within the metastore's managed storage.

## Section 3b: Create Security Groups
Before we can grant permissions, we need to create the groups that will receive those permissions.
### What are Groups?
**Groups** are collections of users that share the same access permissions. Using groups instead of individual users:
- ✅ Simplifies permission management
- ✅ Ensures consistent access across teams
- ✅ Makes auditing easier
- ✅ Follows security best practices
### Persona-Based Groups
We'll create groups for common data team roles:
| Group | Role | Typical Access |
|-------|------|----------------|
| `data_engineers` | Build and maintain pipelines | Full access to Bronze, Silver, Gold |
| `data_analysts` | Create reports and dashboards | Read-only on Silver and Gold |
| `data_scientists` | Build ML models | Read-only on Silver (feature engineering) |
| `business_users` | View business reports | Read-only on specific Gold views |
| `managers` | Oversee operations | Read-only on Gold (high-level metrics) |
### Group Management Notes
**In production:**
- Groups are typically created by workspace admins
- Often synced from corporate identity providers (Azure AD, Okta, etc.)
- Group membership is managed centrally
**For this lab:**
- Groups must be created manually via the Admin Console (not SQL)
- You'll need workspace admin or account admin privileges
- Groups persist across workspace sessions
### 🔧 How to Create Groups Manually
**Step-by-step instructions:**
1. **Navigate to Admin Console:**
   - Click your profile icon (top right)
   - Select **Settings** → **Admin Console**
   - OR append `/settings/workspace/identity-and-access/groups` to your workspace URL
2. **Create Groups:**
   - Click **Groups** tab (left sidebar)
   - Click **Create Group**
   - Create each of the following groups:
     - `data_engineers`
     - `data_analysts`
     - `data_scientists`
     - `business_users`
     - `managers`
   - After creating them, click on each group name, go to "Entitlements" and check all 3 boxes.
3. **Add Yourself to Groups (Optional):**
   - Click on a group name
   - Click **Add Members**
   - Add your user email
   - This allows you to test `is_member()` functions later
**Direct URL Pattern:**
```
https://dbc-XXX.cloud.databricks.com/settings/workspace/identity-and-access/groups
```
(Replace `dbc-XXX` with your unique workspace URL)

**⚠️ Important:** Complete this step before proceeding to the GRANT statements below, as they depend on these groups existing.

In [0]:
# Verify groups exist (run this after creating groups in Admin Console)
print("📋 Checking for required groups...")
print("="*70)

required_groups = [
    "data_engineers",
    "data_analysts",
    "data_scientists",
    "business_users",
    "managers"
]

try:
    all_groups = [row.name for row in spark.sql("SHOW GROUPS").collect()]

    missing_groups = []
    for group in required_groups:
        if group in all_groups:
            print(f"✅ {group} - Found")
        else:
            print(f"❌ {group} - NOT FOUND")
            missing_groups.append(group)

    if missing_groups:
        print(f"\n⚠️  Missing groups: {', '.join(missing_groups)}")
        print(f"⚠️  Please create these groups in the Admin Console before proceeding.")
        print(f"⚠️  Navigate to: [Workspace URL]/settings/workspace/identity-and-access/groups")
    else:
        print(f"\n✅ All required groups exist!")

except Exception as e:
    print(f"⚠️  Could not verify groups: {e}")
    print(f"⚠️  You may need admin privileges to list groups")

print("="*70)

📋 Checking for required groups...
✅ data_engineers - Found
✅ data_analysts - Found
✅ data_scientists - Found
✅ business_users - Found
✅ managers - Found

✅ All required groups exist!


### Checking Your Group Membership
To see which groups you belong to, you can check your current user and group memberships.

**Note:** Adding users to groups is done via the Admin Console UI, not SQL.

In [0]:
# Check current user and their group memberships
current_user_email = spark.sql("SELECT current_user()").collect()[0][0]
print(f"Current user: {current_user_email}")

# Check if you're a member of the key groups
print("\nYour group memberships:")
for group in required_groups:
    try:
        is_member = spark.sql(f"SELECT is_member('{group}')").collect()[0][0]
        status = "✅ Member" if is_member else "❌ Not a member"
        print(f"  {group}: {status}")
    except Exception as e:
        print(f"  {group}: ⚠️  Could not check ({str(e)[:50]})")

Current user: andrew_doublard@outlook.com

Your group memberships:
  data_engineers: ✅ Member
  data_analysts: ✅ Member
  data_scientists: ✅ Member
  business_users: ✅ Member
  managers: ✅ Member


## Section 4: Access Control with GRANT and REVOKE
Unity Catalog provides fine-grained access control.
### Access Levels
- **Catalog level**: All schemas and tables
- **Schema level**: All tables in schema
- **Table level**: Specific tables
### Common Privileges
| Privilege | What It Allows |
|-----------|----------------|
| `SELECT` | Read data |
| `MODIFY` | Insert, update, delete |
| `CREATE` | Create new objects |
| `USAGE` | Access child objects (prerequisite) |
| `ALL PRIVILEGES` | All permissions |
### The USAGE Privilege (Metastore Limitation)
**Important Note**: In Unity Catalog metastore version 1.0, the `USAGE` privilege is not supported on catalogs and schemas.
- In newer metastore versions, `USAGE` would be a prerequisite to access child objects
- For metastore v1.0, you can grant privileges directly (e.g., `SELECT`, `ALL PRIVILEGES`) without `USAGE`
Let's implement persona-based access control.

In [0]:
# Example: Grant full access to data engineers on bronze schema

spark.sql(f"""
    GRANT ALL PRIVILEGES ON SCHEMA {CATALOG_NAME}.{BRONZE_SCHEMA} TO `data_engineers`
""")

print("✅ Granted full access to data_engineers on bronze schema")

✅ Granted full access to data_engineers on bronze schema


---
### 🎯 EXERCISE 2: Grant Permissions for Data Analysts
**Scenario**: Data analysts need read-only access to the Gold schema (business reports).
**Your task**: Write GRANT statements for the `data_analysts` group.
**Requirements:**
1. Grant `SELECT` on the gold schema (read-only permission)
**SQL Syntax:**
```sql
GRANT privilege ON object_type object_name TO `principal`
```
**Hint**: Use `{CATALOG_NAME}` and `{GOLD_SCHEMA}` variables

**Note**: We're skipping USAGE privilege due to metastore v1.0 limitations.

In [0]:
# TODO: Grant read-only access to data analysts on Gold schema

# Grant SELECT on gold schema
spark.sql(f"""
GRANT SELECT ON SCHEMA {CATALOG_NAME}.{GOLD_SCHEMA} TO `data_analysts`
""")

print("Granted read-only access to data_analysts on gold schema")

Granted read-only access to data_analysts on gold schema


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Data Analysts - Gold Read-Only

# spark.sql(f"""
#     GRANT SELECT ON SCHEMA {CATALOG_NAME}.{GOLD_SCHEMA} TO `data_analysts`
# """)

# print("✅ Granted read-only access to data_analysts on gold schema")

---
### 🎯 EXERCISE 3: Grant Permissions for Data Scientists
**Scenario**: Data scientists need read-only access to BOTH Silver and Gold schemas.
**Your task**: Write GRANT statements for the `data_scientists` group.
**Requirements:**
1. Grant `SELECT` on silver schema
2. Grant `SELECT` on gold schema
**Hint**: You'll need 2 GRANT statements total
**Note**: We're skipping USAGE privilege due to metastore v1.0 limitations.

In [0]:
# TODO: Grant read-only access to data scientists on Silver and Gold

# 1. SELECT on silver schema
spark.sql(f"""
          GRANT SELECT ON SCHEMA {CATALOG_NAME}.{SILVER_SCHEMA} TO `data_scientists`
          """)

# 2. SELECT on gold schema
spark.sql(f"""
          GRANT SELECT ON SCHEMA {CATALOG_NAME}.{GOLD_SCHEMA} TO `data_scientists`
          """)

print("Granted read-only access to data_scientists on silver and gold")

Granted read-only access to data_scientists on silver and gold


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Data Scientists - Silver and Gold Read-Only

# spark.sql(f"""
#     GRANT SELECT ON SCHEMA {CATALOG_NAME}.{SILVER_SCHEMA} TO `data_scientists`
# """)

# spark.sql(f"""
#     GRANT SELECT ON SCHEMA {CATALOG_NAME}.{GOLD_SCHEMA} TO `data_scientists`
# """)

# print("✅ Granted read-only access to data_scientists on silver and gold")

In [0]:
# View current grants on bronze schema
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG_NAME}.{BRONZE_SCHEMA}"))

Principal,ActionType,ObjectType,ObjectKey
data_engineers,ALL PRIVILEGES,SCHEMA,cert_prep_catalog.01_bronze


### Revoking Permissions
To revoke permissions, use `REVOKE`:
```sql
REVOKE privilege ON object_type object_name FROM `principal`
```
**Example:**

In [0]:
# Example: Revoke SELECT permission (commented out)
# spark.sql(f"""
#     REVOKE SELECT ON SCHEMA {CATALOG_NAME}.{GOLD_SCHEMA} FROM `data_analysts`
# """)

# Note: Commented to preserve our grants

## Section 5: Serverless Compute
### What is Serverless?
Serverless compute eliminates cluster configuration and management.

**Key characteristics:**
- ⚡ Instant startup (no provisioning)
- 📈 Auto-scaling based on workload
- 💰 Pay only for resources used
- 🔧 Pre-warmed and optimized
### Serverless vs Custom Clusters
| Feature | Serverless | Custom Clusters |
|---------|------------|----------------|
| **Startup Time** | Instant | 3-5 minutes |
| **Configuration** | Automatic | Manual |
| **Scaling** | Auto | Manual/auto rules |
| **Use Case** | Ad-hoc, development | Production, specific configs |
| **Cost** | Per query | Per uptime |
### Best Practices
- ✅ Use for development and exploration
- ✅ Use for ad-hoc queries
- ❌ Not ideal for long-running batch jobs (use custom clusters)

### We're going to use serverless for this lab, as this is the only option in Databricks Free Edition

## Section 6: Summary and Checkpoint
### 🎯 Key Concepts
**1. Unity Catalog Three-Level Namespace**
- `catalog.schema.table` structure
- Enables environment separation and access control
**2. Catalogs, Schemas, and Volumes**
- Hierarchical organization
- Volumes for governed file access
**3. Managed Tables**
- UC controls storage, DROP deletes data
- Best for data exclusively used in Databricks
**4. Access Control**
- GRANT/REVOKE for permissions
- Persona-based patterns (engineers, analysts, scientists)
- Privileges: SELECT, MODIFY, CREATE, USAGE, ALL PRIVILEGES
**5. Serverless Compute**
- Instant startup, auto-scaling
- Best for development and ad-hoc queries
- Community Edition limitations
### ✅ Exam Checklist
Can you:
- [ ] Write three-level namespace references?
- [ ] Create managed tables with saveAsTable()?
- [ ] Write GRANT statements for different personas?
- [ ] Understand the USAGE prerequisite?
- [ ] Describe serverless compute benefits?
### 📚 Next Steps
**Notebook 02** covers:
- Auto Loader for incremental ingestion
- Schema evolution and inference
- Streaming vs batch patterns
---
**🎉 Notebook Complete!** 
You've mastered Unity Catalog basics. Save your work and proceed to Notebook 02.